# Assignment: Steel Plate Fault Classification

**TTK4260 — Multivariate Data Analysis & Machine Learning**  
Department of Engineering Cybernetics, NTNU

---

## Context

In steel manufacturing, surface defects on rolled steel plates must be detected and classified automatically from image-based inspection systems. Each fault is described by geometric, luminosity, and shape features extracted from the defect region. Accurate classification is critical for quality control and process optimisation.

The **Steel Plates Faults** dataset (UCI Machine Learning Repository) contains **1941 observations** with **27 features** and **7 fault classes**:

| Fault Type | Samples | Description |
|:---|:---:|:---|
| Other_Faults | 673 | Miscellaneous defects |
| Bumps | 402 | Surface bumps/protrusions |
| K_Scratch | 391 | K-type scratch patterns |
| Z_Scratch | 190 | Z-type scratch patterns |
| Pastry | 158 | Pastry-like flaking defects |
| Stains | 72 | Surface staining |
| Dirtiness | 55 | Surface contamination |

The 27 features fall into natural groups:

- **Location & geometry:** `X_Minimum`, `X_Maximum`, `Y_Minimum`, `Y_Maximum`
- **Size:** `Pixels_Areas`, `X_Perimeter`, `Y_Perimeter`
- **Luminosity:** `Sum_of_Luminosity`, `Minimum_of_Luminosity`, `Maximum_of_Luminosity`, `Luminosity_Index`
- **Steel properties:** `Length_of_Conveyer`, `TypeOfSteel_A300`, `TypeOfSteel_A400`, `Steel_Plate_Thickness`
- **Shape indices:** `Edges_Index`, `Empty_Index`, `Square_Index`, `Outside_X_Index`, `Edges_X_Index`, `Edges_Y_Index`, `Outside_Global_Index`
- **Derived features:** `LogOfAreas`, `Log_X_Index`, `Log_Y_Index`, `Orientation_Index`, `SigmoidOfAreas`

---

## Objective

Build, evaluate, and compare classification models for steel plate fault detection, applying the full machine-learning pipeline covered in TTK4260.

---

## Instructions

- Fill in the code cells marked with `# YOUR CODE HERE`.
- Answer discussion questions in the markdown cells provided.
- You may add extra cells for exploration, but keep the structure intact.
- Use `random_state=42` wherever applicable for reproducibility.

---
## Part 0: Setup and Data Loading

Download the dataset from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/198/steel+plates+faults) or load it using the `ucimlrepo` package as shown below.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42

In [ ]:
# Option 1: Load from ucimlrepo (install with: pip install ucimlrepo)
from ucimlrepo import fetch_ucirepo

steel = fetch_ucirepo(id=198)
X_raw = steel.data.features
y_raw = steel.data.targets

# The target is stored as 7 binary columns — convert to a single label
y_labels = y_raw.idxmax(axis=1)

# Combine into a single DataFrame for exploration
data = pd.concat([X_raw, y_labels.rename('Fault_Type')], axis=1)

print(f'Dataset shape: {data.shape}')
data.head()

---
## Part 1: Exploratory Data Analysis

### Task 1.1 — Basic Inspection

Print the data types, check for missing values, and display summary statistics. Note which features are continuous, which are binary (`TypeOfSteel_A300`, `TypeOfSteel_A400`), and whether any features appear to be pre-engineered (e.g., `LogOfAreas`, `SigmoidOfAreas`).

In [ ]:
# YOUR CODE HERE


### Task 1.2 — Class Distribution

Plot the distribution of the 7 fault types as a bar chart. Comment on the class imbalance and its potential consequences for training and evaluation.

In [ ]:
# YOUR CODE HERE


**Discussion:** *(Comment on class balance. Which classes might be hard to learn?)*


### Task 1.3 — Feature Distributions by Class

Choose 3–4 features (e.g., one from each feature group: geometry, luminosity, shape) and create box plots or violin plots grouped by fault type. Are there features that clearly separate some classes?

In [ ]:
# YOUR CODE HERE


**Discussion:** *(Which features show visible separation between classes?)*


### Task 1.4 — Correlation Analysis

Compute and plot the correlation matrix of the 27 features as a heatmap. Identify highly correlated pairs (|r| > 0.8). What does this suggest about redundancy or multicollinearity?

In [ ]:
# YOUR CODE HERE


**Discussion:** *(Identify correlated feature pairs and discuss implications.)*


---
## Part 2: Data Preparation

### Task 2.1 — Feature Matrix and Target Vector

Separate the data into a feature matrix `X` (27 features) and target vector `y` (fault type labels). Encode the target labels as integers using `LabelEncoder` for compatibility with all classifiers.

In [ ]:
# YOUR CODE HERE


### Task 2.2 — Train/Test Split

Split into 80% training and 20% test sets using **stratified** splitting. Verify that class proportions are preserved in both sets.

In [ ]:
# YOUR CODE HERE


### Task 2.3 — Feature Scaling

Standardise the features (zero mean, unit variance). Fit the scaler on the training set only, then transform both training and test sets.

In [ ]:
# YOUR CODE HERE


---
## Part 3: Feature Engineering & Selection

### Task 3.1 — Feature Engineering

Create at least **three** new features that combine existing ones in physically meaningful ways. Examples:

- **Defect width:** `X_Maximum - X_Minimum`
- **Defect height:** `Y_Maximum - Y_Minimum`
- **Aspect ratio:** `(X_Maximum - X_Minimum) / (Y_Maximum - Y_Minimum + 1)`
- **Luminosity range:** `Maximum_of_Luminosity - Minimum_of_Luminosity`
- **Perimeter ratio:** `X_Perimeter / (Y_Perimeter + 1)`
- **Area-to-perimeter ratio:** `Pixels_Areas / (X_Perimeter + Y_Perimeter + 1)`

Add these to both training and test feature matrices. Remember to standardise the new features as well.

In [ ]:
# YOUR CODE HERE


### Task 3.2 — Feature Selection

Apply **two different** feature selection methods (e.g., mutual information ranking and tree-based feature importance). Rank the features and identify the top 10 and bottom 5. Do your engineered features appear among the top?

In [ ]:
# YOUR CODE HERE


**Discussion:** *(Which features are most informative? Do the two methods agree? Do engineered features help?)*


---
## Part 4: Logistic Regression

### Task 4.1 — Baseline Logistic Regression

Train a multinomial logistic regression model with default parameters. Report the training and test accuracy.

In [ ]:
# YOUR CODE HERE


### Task 4.2 — Regularisation Tuning

Use **5-fold stratified cross-validation** to tune the regularisation parameter `C`. Try at least 5 values spanning several orders of magnitude (e.g., 0.001 to 100). Plot mean CV accuracy ± standard deviation vs `C` and select the best.

In [ ]:
# YOUR CODE HERE


### Task 4.3 — Evaluate Logistic Regression

For the best model, display:
1. **Confusion matrix** as a heatmap (use class names as labels)
2. **Classification report** (precision, recall, F1 per class)

Which fault types are hardest for logistic regression?

In [ ]:
# YOUR CODE HERE


**Discussion:** *(Which classes have low recall or precision? Why might a linear model struggle here?)*


---
## Part 5: Support Vector Machine

### Task 5.1 — SVM with RBF Kernel

Train an SVM with RBF kernel. Use cross-validation (or `GridSearchCV`) to tune `C` and `gamma`. Report the best hyperparameters and cross-validation accuracy.

*Hint:* Try `C` in {0.1, 1, 10, 100} and `gamma` in {'scale', 0.01, 0.1, 1}.

In [ ]:
# YOUR CODE HERE


### Task 5.2 — Evaluate SVM

Display the confusion matrix and classification report for the tuned SVM on the test set. Compare per-class performance with logistic regression — which classes improved?

In [ ]:
# YOUR CODE HERE


**Discussion:** *(How does SVM compare to logistic regression? Where does the nonlinear boundary help?)*


---
## Part 6: Decision Tree & Ensemble Methods

### Task 6.1 — Decision Tree

Train a decision tree classifier. Use cross-validation to tune `max_depth` (try values from 2 to 20). Report:
- Training accuracy and test accuracy for the best tree
- Whether you see signs of overfitting (training ≫ test accuracy)

In [ ]:
# YOUR CODE HERE


### Task 6.2 — Random Forest

Train a Random Forest. Experiment with `n_estimators` (50, 100, 200) and `max_depth`. Display the **feature importances** as a horizontal bar chart.

In [ ]:
# YOUR CODE HERE


### Task 6.3 — Tree vs Forest Comparison

Display the confusion matrix for the Random Forest. Compare with the single decision tree — is there evidence of variance reduction?

In [ ]:
# YOUR CODE HERE


**Discussion:** *(Single tree vs Random Forest: what improved? Which fault types benefit most from ensembling?)*


---
## Part 7: Independent Component Analysis (ICA)

### Task 7.1 — Apply ICA

The 27 features come from image processing of the defect region — geometry, luminosity statistics, and shape indices are extracted from the same underlying image. These measured features may be **mixtures of a smaller number of latent factors** (e.g., defect size, defect shape, surface reflectivity, steel type effects).

Apply ICA (e.g., `sklearn.decomposition.FastICA`) to the standardised training features. Extract a reasonable number of independent components (e.g., 10–15). Transform both training and test sets.

In [ ]:
# YOUR CODE HERE


### Task 7.2 — Visualise ICA Components

Create a 2D scatter plot of the first two independent components, coloured by fault type. Do you see any class separation?

In [ ]:
# YOUR CODE HERE


### Task 7.3 — Classification on ICA Features

Train a classifier of your choice (e.g., logistic regression) on the ICA-transformed features. Compare the test accuracy to the same classifier trained on the original features. Does ICA help?

In [ ]:
# YOUR CODE HERE


**Discussion:** *(Does ICA improve or hurt performance? What does this tell you about whether the original features are truly "mixed" signals versus direct measurements?)*


---
## Part 8: Cross-Validation & Performance Metrics

### Task 8.1 — Stratified K-Fold Comparison

Using **5-fold stratified cross-validation** on the training set, compute the mean and standard deviation of accuracy for all your models:

1. Logistic Regression (best `C`)
2. SVM (best `C`, `gamma`)
3. Decision Tree (best `max_depth`)
4. Random Forest (best settings)

Present the results in a bar chart with error bars.

In [ ]:
# YOUR CODE HERE


### Task 8.2 — Metrics for Imbalanced Data

Overall accuracy can be misleading when classes are imbalanced (Dirtiness has 55 samples vs Other_Faults with 673). For your **best model**, report:

- **Macro-averaged** precision, recall, and F1
- **Weighted-averaged** precision, recall, and F1

Explain the difference between macro and weighted averaging and which is more informative for this problem.

In [ ]:
# YOUR CODE HERE


**Discussion:** *(Macro vs weighted: which better reflects model quality in a manufacturing context where all fault types matter?)*


---
## Part 9: Summary & Reflection

### Task 9.1 — Summary Table

Fill in the table below with your results.

| Model | Test Accuracy | Macro F1 | Weighted F1 | Key Hyperparameters |
|:---|:---:|:---:|:---:|:---|
| Logistic Regression | | | | |
| SVM (RBF) | | | | |
| Decision Tree | | | | |
| Random Forest | | | | |
| ICA + Classifier | | | | |

### Task 9.2 — Final Model Recommendation

Select the model you would deploy for automated steel plate inspection. Justify your choice by considering:
- Overall accuracy and per-class performance
- Performance on rare fault types (Dirtiness, Stains)
- Computational cost and interpretability
- Robustness (low variance across folds)

**Your answer:** *(Write here)*


### Task 9.3 — Reflection

In a short paragraph, reflect on:
1. Where did **logistic regression** struggle and why? What does this reveal about the decision boundaries?
2. Did **feature engineering** or **feature selection** meaningfully improve any model?
3. What was the most important lesson about choosing **performance metrics** for imbalanced data?
4. What would you try next — e.g., gradient boosting, oversampling (SMOTE), neural networks?

**Your answer:** *(Write here)*
